# Notebook 2 — LLM Transaction Enrichment

**Purpose:** Batch-categorise production transactions using AWS Bedrock and evaluate accuracy against known labels from Notebook 1.

**Inputs:**
| File | Description |
|---|---|
| `outputs/taxonomy_master.json` | Structured taxonomy from Notebook 1 |
| `outputs/taxonomy_master.csv` | Master taxonomy with spend types |
| Production DB | Live transactions via `dbx_query_df` |

**Outputs:**
| File | Description |
|---|---|
| `outputs/enrichment_results.csv` | Per-transaction LLM predictions vs ground truth |
| `outputs/enrichment_accuracy.csv` | Accuracy metrics by category and provider |
| `experiments/enrichment_*.csv` | Cost and token tracking from ExperimentTracker |

---
## 0. Setup

In [ ]:
import boto3
import json
import os
import time
import pandas as pd
import databricks.sql as sql
from pathlib import Path
from datetime import datetime
from typing import Optional
from dotenv import load_dotenv

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
Path("experiments").mkdir(exist_ok=True)

print("Setup complete")

---
## 1. Bedrock Client & Experiment Tracker

Copied from the Bedrock experimentation framework. Single place to update if models or pricing change.

In [ ]:
# --- Model config ---
MODELS = {
    "claude-haiku":      "anthropic.claude-3-haiku-20240307-v1:0",
    "claude-sonnet":     "anthropic.claude-3-sonnet-20240229-v1:0",
    "claude-sonnet-3.5": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    "nova-micro":        "amazon.nova-micro-v1:0",
    "nova-lite":         "amazon.nova-lite-v1:0",
    "nova-pro":          "amazon.nova-pro-v1:0",
}

PRICING = {
    "anthropic.claude-3-haiku-20240307-v1:0":     {"input": 0.25,  "output": 1.25},
    "anthropic.claude-3-sonnet-20240229-v1:0":    {"input": 3.00,  "output": 15.00},
    "anthropic.claude-3-5-sonnet-20241022-v2:0":  {"input": 3.00,  "output": 15.00},
    "amazon.nova-micro-v1:0":                     {"input": 0.035, "output": 0.14},
    "amazon.nova-lite-v1:0":                      {"input": 0.06,  "output": 0.24},
    "amazon.nova-pro-v1:0":                       {"input": 0.80,  "output": 3.20},
}

CONTEXT_LIMITS = {
    "anthropic.claude-3-haiku-20240307-v1:0":    200_000,
    "anthropic.claude-3-sonnet-20240229-v1:0":   200_000,
    "anthropic.claude-3-5-sonnet-20241022-v2:0": 200_000,
    "amazon.nova-micro-v1:0":                    128_000,
    "amazon.nova-lite-v1:0":                     300_000,
    "amazon.nova-pro-v1:0":                      300_000,
}

In [ ]:
class BedrockClient:
    def __init__(self, model: str = "claude-haiku", region: str = "ap-southeast-2"):
        self.model_alias = model
        self.model_id    = MODELS[model]
        self.provider    = "anthropic" if "anthropic" in self.model_id else "amazon"
        self.region = region

    def invoke(self, prompt: str, system: str = None,
               max_tokens: int = 4096, temperature: float = 0.0,
               max_retries: int = 3) -> dict:
        bedrock = boto3.client("bedrock-runtime", region_name=self.region)               
        body = self._build_body(prompt, system, min(max_tokens, 4096), temperature)
        for attempt in range(max_retries):
            try:
                resp   = bedrock.invoke_model(
                    modelId     = self.model_id,
                    contentType = "application/json",
                    accept      = "application/json",
                    body        = json.dumps(body)
                )

                result = self._parse_response(json.loads(resp["body"].read()))
                result["model"]    = self.model_alias
                result["model_id"] = self.model_id
                result["cost"]     = self._calculate_cost(result["input_tokens"], result["output_tokens"])
                return result
            except bedrock.exceptions.ThrottlingException:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    raise

    def _build_body(self, prompt, system, max_tokens, temperature):
        if self.provider == "anthropic":
            body = {"anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": max_tokens, "temperature": temperature,
                    "messages": [{"role": "user", "content": prompt}]}
            if system:
                body["system"] = system
        else:
            body = {"messages": [{"role": "user", "content": [{"text": prompt}]}],
                    "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature}}
            if system:
                body["system"] = [{"text": system}]
        return body

    def _parse_response(self, data):
        if self.provider == "anthropic":
            return {"content":       data["content"][0]["text"],
                    "input_tokens":  data["usage"]["input_tokens"],
                    "output_tokens": data["usage"]["output_tokens"],
                    "total_tokens":  data["usage"]["input_tokens"] + data["usage"]["output_tokens"]}
        else:
            return {"content":       data["output"]["message"]["content"][0]["text"],
                    "input_tokens":  data["usage"]["inputTokens"],
                    "output_tokens": data["usage"]["outputTokens"],
                    "total_tokens":  data["usage"]["totalTokens"]}

    def _calculate_cost(self, input_tokens, output_tokens):
        p = PRICING.get(self.model_id, {"input": 0, "output": 0})
        return (input_tokens / 1_000_000) * p["input"] + (output_tokens / 1_000_000) * p["output"]

    def get_context_limit(self):
        return CONTEXT_LIMITS.get(self.model_id, 200_000)

    def __repr__(self):
        return f"BedrockClient(model='{self.model_alias}')"


def get_client(model: str = "claude-haiku") -> BedrockClient:
    return BedrockClient(model=model)

print("BedrockClient ready. Available models:", list(MODELS.keys()))

In [ ]:
class ExperimentTracker:
    def __init__(self, name: str, budget_usd: float = None):
        self.name        = name
        self.timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiments = []
        self.budget_usd  = budget_usd
        self._warned     = False

    def log(self, response: dict, prompt_preview: str = "", notes: str = ""):
        cost = response.get("cost", 0)
        if self.budget_usd:
            total = self.total_cost() + cost
            if total > self.budget_usd:
                raise ValueError(f"BUDGET EXCEEDED: ${total:.4f} > ${self.budget_usd:.4f}")
            if not self._warned and total >= self.budget_usd * 0.8:
                print(f"WARNING: 80% of budget used (${total:.4f} / ${self.budget_usd:.4f})")
                self._warned = True
        self.experiments.append({
            "timestamp":        datetime.now().isoformat(),
            "model":            response.get("model", ""),
            "input_tokens":     response.get("input_tokens", 0),
            "output_tokens":    response.get("output_tokens", 0),
            "total_tokens":     response.get("total_tokens", 0),
            "cost_usd":         cost,
            "prompt_preview":   prompt_preview[:100],
            "response_preview": response.get("content", "")[:100],
            "notes":            notes,
        })

    def summary(self)    -> pd.DataFrame: return pd.DataFrame(self.experiments)
    def total_cost(self) -> float:        return sum(e["cost_usd"] for e in self.experiments)
    def total_tokens(self) -> int:        return sum(e["total_tokens"] for e in self.experiments)

    def budget_status(self):
        spent = self.total_cost()
        if self.budget_usd:
            print(f"Budget: ${spent:.4f} / ${self.budget_usd:.4f} ({spent/self.budget_usd*100:.1f}%)")
        else:
            print(f"Total spent: ${spent:.4f} (no budget set)")

    def save(self, path: str = None) -> str:
        p = path or f"experiments/{self.name}_{self.timestamp}.csv"
        self.summary().to_csv(p, index=False)
        print(f"Saved: {p}")
        return p

print("ExperimentTracker ready")

---
## 2. DB Connection

Shared `dbx_query_df` helper — same pattern as Notebook 1.

In [ ]:
load_dotenv(dotenv_path=r"/home/sagemaker-user/swtest1/llm_poc/.env", override=True)
os.environ.pop("AWS_BEARER_TOKEN_BEDROCK", None) 
print("HOST:", os.getenv("DATABRICKS_SERVER_HOSTNAME"))
print("HTTP PATH:", os.getenv("DATABRICKS_HTTP_PATH"))
print("TOKEN set?", bool(os.getenv("DATABRICKS_TOKEN")))

In [ ]:
def dbx_query_df(query: str, *, catalog=None, schema=None) -> pd.DataFrame:
    with sql.connect(
        server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
        http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
        access_token    = os.getenv("DATABRICKS_TOKEN"),
        catalog         = catalog or os.getenv("DATABRICKS_CATALOG"),
        schema          = schema  or os.getenv("DATABRICKS_SCHEMA"),
        use_cloud_fetch = True,
    ) as conn, conn.cursor() as cur:
        cur.execute(query)
        return cur.fetchall_arrow().to_pandas(types_mapper=pd.ArrowDtype)

print("DB connection helper ready")

---
## 3. Load Taxonomy Context

Load `taxonomy_master.json` from Notebook 1 as the LLM system prompt context.
Also load `taxonomy_master.csv` for valid key validation.

In [ ]:
# Load taxonomy outputs from Notebook 1
with open(OUTPUT_DIR / "taxonomy_master.json") as f:
    taxonomy_json = json.load(f)

taxonomy_master = pd.read_csv(OUTPUT_DIR / "taxonomy_master.csv")

# Load spend/non-spend decision rules from markdown
with open("assets/rules_sonnet-3.5_20260121.md") as f:
    DECISION_RULES = f.read()

# Build set of valid canonical keys for response validation
VALID_KEYS = set(taxonomy_master["category_key"].str.upper())

# Serialise taxonomy JSON as context string for LLM
TAXONOMY_CONTEXT = json.dumps(taxonomy_json, indent=2)

est_tokens = len(TAXONOMY_CONTEXT) // 4
print(f"Taxonomy loaded: {len(VALID_KEYS)} valid keys")
print(f"Context size   : ~{est_tokens:,} tokens")
print(f"Spend keys     : {len(taxonomy_json.get('spend', {}))} groups")
print(f"Non-spend keys : {len(taxonomy_json.get('non_spend', {}))} groups")

---
## 4. Prompt Template

System prompt injects the full taxonomy. User prompt provides per-transaction fields.
Adjust the template here to experiment with different prompting strategies.

In [ ]:
SYSTEM_PROMPT = f"""You are a financial transaction categorisation engine.

Your task is to assign a single base_category_key to each transaction.
Respond with ONLY the category key — no explanation, no punctuation, no extra text.

Rules:
- Use ONLY keys that exist in the taxonomy below
- Follow the spend/non-spend decision rules precisely
- Prefer more specific keys over generic ones (e.g. GROCERIES over GENERAL_MERCHANDISE)
- When merchant_name is present and matches a spend category, prefer that over description patterns
- If truly ambiguous, return UNCATEGORISED

=== SPEND / NON-SPEND DECISION RULES ===
{DECISION_RULES}
=== END RULES ===

=== TAXONOMY ===
{TAXONOMY_CONTEXT}
=== END TAXONOMY ==="""


def build_user_prompt(row: dict) -> str:
    """Build per-transaction prompt from DB row fields."""
    return f"""Categorise this transaction:

provider          : {row.get('provider_name', '')}
transaction_type  : {row.get('transaction_type', '')}
account_type      : {row.get('account_type', '')}
amount            : {row.get('amount', '')}
original_description: {row.get('original_description', '')}
merchant_name     : {row.get('merchant_name', '')}
frollo_merchant   : {row.get('frollo_merchant', '')}
mcc               : {row.get('merchant_category_code', '')}
cdr_biller_code   : {row.get('cdr_biller_code', '')}
cdr_biller_name   : {row.get('cdr_biller_name', '')}

Respond with only the category key."""


print(f"System prompt  : ~{len(SYSTEM_PROMPT)//4:,} tokens")
print(f"User prompt    : ~{len(build_user_prompt({}))//4} tokens (empty row)")

---
## 5. Pull Sample Transactions from Production

Pull a stratified sample with known ground-truth labels for accuracy evaluation.
Adjust `SAMPLE_DATE_FROM`, `SAMPLE_DATE_TO`, and `TARGET_PER_PROVIDER` as needed.

In [ ]:
SAMPLE_DATE_FROM    = "2025-11-01"
SAMPLE_DATE_TO      = "2025-12-31"
TARGET_PER_PROVIDER = 200    # keep small for initial run — scale up once validated

df_sample = dbx_query_df(f"""
    SELECT
        t.id                            AS transaction_id,
        pa.user_id,
        p.name                          AS provider_name,
        p.aggregator,
        t.account_id,
        t.container,
        a.account_type,
        t.transaction_date,
        t.original_description,
        t.amount,
        t.base_category_key,
        t.transaction_type,
        t.cdr_merchant_category_code    AS merchant_category_code,
        t.cdr_biller_code,
        t.cdr_biller_name,
        t.merchant_name,
        m.name                          AS frollo_merchant
    FROM prod.public.accounts_silver_view a
    INNER JOIN prod.public.provider_accounts_silver_view pa ON pa.id = a.provider_account_id
    INNER JOIN prod.public.providers_silver_view p ON p.id = pa.provider_id
    INNER JOIN prod.public.transactions_silver_view t ON t.account_id = a.id
    INNER JOIN prod.public.merchants_silver_view m ON m.id = t.merchant_id
    WHERE p.ideas_enrich_v2 = True
      AND t.base_category_key IS NOT NULL
      AND t.created_at BETWEEN DATE '{SAMPLE_DATE_FROM}' AND DATE '{SAMPLE_DATE_TO}'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY p.name
        ORDER BY RAND(42)
    ) <= {TARGET_PER_PROVIDER}
""")

# Normalise ground-truth labels to uppercase
df_sample["ground_truth"] = df_sample["base_category_key"].str.strip().str.upper()

print(f"Sample rows      : {len(df_sample):,}")
print(f"Providers        : {df_sample['provider_name'].nunique()}")
print(f"Distinct GT keys : {df_sample['ground_truth'].nunique()}")
print(f"\nGround truth distribution (top 15):")
print(df_sample["ground_truth"].value_counts().head(15).to_string())

In [ ]:
df_sample.to_parquet("data/df_sample_small.parquet", index=False)

In [ ]:
df_sample = pd.read_parquet("data/df_sample_small.parquet")
df_sample.shape

---
## 6. Test Connection & Single Transaction

Run one transaction before committing to the full batch.

In [ ]:
client = get_client("claude-haiku")  # start cheap
print(f"Model          : {client}")
print(f"Context limit  : {client.get_context_limit():,} tokens\n")

# Test with first row
test_row    = df_sample.iloc[0].to_dict()
test_prompt = build_user_prompt(test_row)

response = client.invoke(test_prompt, system=SYSTEM_PROMPT, max_tokens=20, temperature=0.0)

print(f"Ground truth   : {test_row['ground_truth']}")
print(f"LLM prediction : {response['content'].strip().upper()}")
print(f"Match          : {test_row['ground_truth'] == response['content'].strip().upper()}")
print(f"\nTokens         : {response['input_tokens']} in / {response['output_tokens']} out")
print(f"Cost per txn   : ${response['cost']:.6f}")
print(f"Est. batch cost: ${response['cost'] * len(df_sample):.4f} ({len(df_sample)} rows)")

---
## 7. Batch Enrichment

Run the full sample. Results written incrementally to `enrichment_results.csv`.
Set `BUDGET_USD` before running — the tracker will hard-stop if exceeded.

In [ ]:
BUDGET_USD    = 5.00   # adjust based on single-transaction cost estimate above
RATE_LIMIT_N  = 100    # pause every N requests
RATE_LIMIT_S  = 1      # seconds to pause

tracker = ExperimentTracker("enrichment", budget_usd=BUDGET_USD)
results = []

for i, row in df_sample.iterrows():
    try:
        prompt   = build_user_prompt(row.to_dict())
        response = client.invoke(prompt, system=SYSTEM_PROMPT, max_tokens=20, temperature=0.0)
        pred     = response["content"].strip().upper()

        results.append({
            "transaction_id":    row["transaction_id"],
            "provider_name":     row["provider_name"],
            "transaction_type":  row["transaction_type"],
            "account_type":      row["account_type"],
            "amount":            row["amount"],
            "original_description": row["original_description"],
            "merchant_name":     row["merchant_name"],
            "ground_truth":      row["ground_truth"],
            "llm_prediction":    pred,
            "match":             row["ground_truth"] == pred,
            "valid_key":         pred in VALID_KEYS,
            "cost":              response["cost"],
            "input_tokens":      response["input_tokens"],
            "output_tokens":     response["output_tokens"],
        })

        tracker.log(response,
                    prompt_preview=row["original_description"][:80],
                    notes=f"row {len(results)}/{len(df_sample)}")

        # Rate limit pause
        if len(results) % RATE_LIMIT_N == 0:
            tracker.budget_status()
            time.sleep(RATE_LIMIT_S)

    except ValueError as e:
        print(f"Stopped at row {len(results)}: {e}")
        break
    except Exception as e:
        print(f"Error on transaction {row.get('transaction_id', i)}: {e}")
        results.append({
            "transaction_id": row["transaction_id"],
            "ground_truth":   row["ground_truth"],
            "llm_prediction": "ERROR",
            "match":          False,
            "valid_key":      False,
            "error":          str(e),
        })
        continue

df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_DIR / "enrichment_results.csv", index=False)
tracker.save()

print(f"\nBatch complete: {len(df_results):,} rows processed")
tracker.budget_status()

---
## 8. Accuracy Evaluation

Overall accuracy, invalid key rate, and breakdown by category and provider.

In [ ]:
# --- Overall metrics ---
total        = len(df_results)
correct      = df_results["match"].sum()
invalid_keys = (~df_results["valid_key"]).sum()
errors       = df_results.get("error", pd.Series(dtype=str)).notna().sum()

print("=" * 50)
print("OVERALL ACCURACY")
print("=" * 50)
print(f"Total processed  : {total:,}")
print(f"Correct          : {correct:,}  ({correct/total*100:.1f}%)")
print(f"Incorrect        : {total-correct:,}  ({(total-correct)/total*100:.1f}%)")
print(f"Invalid keys     : {invalid_keys:,}  ({invalid_keys/total*100:.1f}%)")
print(f"Errors           : {errors:,}")
print(f"Total cost       : ${df_results['cost'].sum():.4f}")
print(f"Cost per txn     : ${df_results['cost'].mean():.6f}")

In [ ]:
# --- Accuracy by ground-truth category ---
cat_acc = (
    df_results.groupby("ground_truth")
    .agg(
        total   = ("match", "count"),
        correct = ("match", "sum"),
    )
    .assign(accuracy=lambda x: (x["correct"] / x["total"] * 100).round(1))
    .sort_values("accuracy")
    .reset_index()
)

print("\nAccuracy by category (worst first):")
print(cat_acc.to_string(index=False))

In [ ]:
# --- Accuracy by provider ---
prov_acc = (
    df_results.groupby("provider_name")
    .agg(
        total   = ("match", "count"),
        correct = ("match", "sum"),
    )
    .assign(accuracy=lambda x: (x["correct"] / x["total"] * 100).round(1))
    .sort_values("accuracy")
    .reset_index()
)

print("Accuracy by provider (worst first):")
print(prov_acc.to_string(index=False))

In [ ]:
# --- Confusion: where did the LLM go wrong? ---
# Most common mismatches
errors_df = df_results[~df_results["match"]].copy()

confusion = (
    errors_df.groupby(["ground_truth", "llm_prediction"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(f"\nTop mismatches (ground_truth → llm_prediction):")
print(confusion.head(20).to_string(index=False))

In [ ]:
# --- Export accuracy report ---
cat_acc.to_csv(OUTPUT_DIR / "enrichment_accuracy_by_category.csv", index=False)
prov_acc.to_csv(OUTPUT_DIR / "enrichment_accuracy_by_provider.csv", index=False)
confusion.to_csv(OUTPUT_DIR / "enrichment_confusion.csv", index=False)

print("Exported:")
print("  outputs/enrichment_accuracy_by_category.csv")
print("  outputs/enrichment_accuracy_by_provider.csv")
print("  outputs/enrichment_confusion.csv")

---
## 9. Cost Projection

Extrapolate from sample cost to production-scale estimates.

In [ ]:
avg_cost_per_txn = df_results["cost"].mean()
avg_tokens_per_txn = df_results["input_tokens"].mean() + df_results["output_tokens"].mean()

print(f"Avg cost per transaction : ${avg_cost_per_txn:.6f}")
print(f"Avg tokens per transaction: {avg_tokens_per_txn:.0f}")
print()
print(f"{'Scale':<20} {'Est. Cost':>12}")
print("-" * 34)
for n in [1_000, 10_000, 100_000, 500_000, 1_000_000]:
    print(f"{n:>20,}  ${avg_cost_per_txn * n:>10.2f}")

---
## 10. Model Comparison (Optional)

Re-run a small subset against a second model to compare accuracy vs cost.
Uncomment and set `COMPARE_MODEL` to run.

In [ ]:
# COMPARE_MODEL  = "claude-sonnet-3.5"  # upgrade candidate
# COMPARE_SAMPLE = 100                  # rows to re-run
# 
# client2  = get_client(COMPARE_MODEL)
# tracker2 = ExperimentTracker(f"enrichment_{COMPARE_MODEL}", budget_usd=2.00)
# results2 = []
# 
# for _, row in df_sample.head(COMPARE_SAMPLE).iterrows():
#     resp = client2.invoke(build_user_prompt(row.to_dict()),
#                           system=SYSTEM_PROMPT, max_tokens=20, temperature=0.0)
#     pred = resp["content"].strip().upper()
#     results2.append({"ground_truth": row["ground_truth"],
#                      "llm_prediction": pred,
#                      "match": row["ground_truth"] == pred,
#                      "cost": resp["cost"]})
#     tracker2.log(resp)
# 
# df2 = pd.DataFrame(results2)
# df1_sub = df_results.head(COMPARE_SAMPLE)
# 
# print(f"{'Model':<25} {'Accuracy':>10} {'Avg Cost':>12}")
# print("-" * 50)
# print(f"{client.model_alias:<25} {df1_sub['match'].mean()*100:>9.1f}%  ${df1_sub['cost'].mean():>10.6f}")
# print(f"{COMPARE_MODEL:<25} {df2['match'].mean()*100:>9.1f}%  ${df2['cost'].mean():>10.6f}")
# tracker2.save()

print("Model comparison cell ready — uncomment COMPARE_MODEL to run")